## 2026 EY AI & Data Challenge - Landsat Data Extraction Notebook

This notebook demonstrates Landsat data extraction and the creation of an output file to be used by the benchmark notebook. The baseline data is [Landsat Collection 2 Level 2](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2) data from the MS Planetary Computer catalog.

**Caution**... This notebook requires significant execution time as there are 9,319 data points (unique locations and times) used for data extraction from the Landsat archive. The code takes about 7 hours to run to completion on a typical laptop computer with a typical internet connection. Lower execution times are likely possible with optimization of the data extraction process and the use of cloud computing services.


### Load In Dependencies
The following code installs the required Python libraries (found in the requirements.txt file) in the Snowflake environment to allow successful execution of the remaining notebook code. After running this code for the first time, it is required to “restart” the kernal so the Python libraries are available in the environment. This is done by selecting the “Connected” menu above the notebook (next to “Run all”) and selecting the “restart kernal” link. Subsequent runs of the notebook do not require this “restart” process. 

In [ ]:
!pip install uv
!uv pip install  -r requirements.txt 

In [1]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os

### Extracting Landsat Data Using API Calls

The API-based method allows us to efficiently access **Landsat** data for specific coordinates and time periods, ensuring scalability and reproducibility of the process.

Through the API, we can query individual bands or compute indices like **NDMI** on the fly. This approach reduces storage requirements and simplifies data preprocessing, making it ideal for large-scale environmental and water quality analysis.

The **compute_Landsat_values** function extracts Landsat surface reflectance values for specific sampling locations using a 100 m focal buffer around each point. For each location:

- A bounding box (bbox) is created around the latitude and longitude coordinates.
- The Microsoft Planetary Computer API is queried for Landsat-8 Level-2 surface reflectance imagery within the date range.
- The nearest low-cloud (<10% cloud cover) scene is selected, and the specified bands (**green**, **nir08**, **swir16**, **swir22**) are loaded.
- Median values of the pixels within the bounding box are computed to reduce the effect of noise or outliers.

**Why the buffer value is 0.00089831**

We want a ~100 m buffer around each point.  
At the equator, 1 degree ≈ 110 km.

Therefore, the degree equivalent of 100 m is:

*buffer_deg ≈ 100 m / 110,000 m per degree ≈ 0.00089831*

This value ensures that the buffer approximately matches the pixel resolution of Landsat imagery, capturing a ~100 m area around each sampling location.


In [2]:
tqdm.pandas()

def compute_Landsat_values(row):

    lat = row['Latitude']
    lon = row['Longitude']
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

    bbox_size = 0.0008
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2
    ]

    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )

    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime="2010-09-30/2016-01-01",
        query={"eo:cloud_cover": {"lt": 10}},
    )

    items = search.item_collection()

    if not items:
        return pd.Series({
            "blue": np.nan,
            "green": np.nan,
            "red": np.nan,
            "nir": np.nan,
            "swir16": np.nan,
            "swir22": np.nan,
        })

    try:
        sample_date_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")

        items = sorted(
            items,
            key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc)
        )

        selected_item = pc.sign(items[0])

        bands_of_interest = [
            "blue", "green", "red",
            "nir08", "swir16", "swir22"
        ]

        data = stac_load(
            [selected_item],
            bands=bands_of_interest,
            bbox=bbox
        ).isel(time=0)

        blue = data["blue"].astype("float")
        green = data["green"].astype("float")
        red = data["red"].astype("float")
        nir = data["nir08"].astype("float")
        swir16 = data["swir16"].astype("float")
        swir22 = data["swir22"].astype("float")

        median_blue = float(blue.median(skipna=True).values)
        median_green = float(green.median(skipna=True).values)
        median_red = float(red.median(skipna=True).values)
        median_nir = float(nir.median(skipna=True).values)
        median_swir16 = float(swir16.median(skipna=True).values)
        median_swir22 = float(swir22.median(skipna=True).values)

        # Replace zeros
        vals = [
            median_blue, median_green, median_red,
            median_nir, median_swir16, median_swir22
        ]
        vals = [v if v != 0 else np.nan for v in vals]
        median_blue, median_green, median_red, median_nir, median_swir16, median_swir22 = vals


        return pd.Series({
            "blue": median_blue,
            "green": median_green,
            "red": median_red,
            "nir": median_nir,
            "swir16": median_swir16,
            "swir22": median_swir22,
        })

    except Exception:
        return pd.Series({
            "blue": np.nan,
            "green": np.nan,
            "red": np.nan,
            "nir": np.nan,
            "swir16": np.nan,
            "swir22": np.nan,
        })


### Extracting features for the training dataset

In [3]:
Water_Quality_df=pd.read_csv('water_quality_training_dataset.csv')
#display(Water_Quality_df.head())

In [4]:
train_features_path = "landsat_extraction_training.csv"
batch_size = 50

if os.path.exists(train_features_path):
    existing = pd.read_csv(train_features_path)
    processed_ids = set(existing.index)  # o mejor usar un ID único si lo tienes
else:
    existing = pd.DataFrame()
    processed_ids = set()


def safe_compute(row):
    try:
        return compute_Landsat_values(row)
    except Exception as e:
        print(f"⚠ Error in row {row.name}: {e}")
        return None

results = []

for start in range(0, len(Water_Quality_df), batch_size):
    end = start + batch_size
    batch = Water_Quality_df.iloc[start:end]

    print(f"🚀 Processing rows {start} to {end}")

    for idx, row in batch.iterrows():
        result = safe_compute(row)
        if result is not None:
            pd.DataFrame([result]).to_csv(
                train_features_path,
                mode='a',
                header=not os.path.exists(train_features_path),
                index=False
            )

    print("✅ Batch saved")


In [ ]:
landsat_train_features = pd.read_csv('landsat_extraction_training.csv')

In [ ]:
# Extract band values from Landsat for training dataset
'''
print("🚀 Running Landsat feature extraction for training data...")
landsat_train_features = Water_Quality_df_20.progress_apply(compute_Landsat_values, axis=1)
landsat_train_features.to_csv(train_features_path, index=False)
'''

In [7]:
def create_indices (dset):

    band = dset.copy()
    # Create indices: NDMI and MNDWI
    eps = 1e-10
    band['NDMI'] = (band['nir'] - band['swir16']) / (band['nir'] + band['swir16'] + eps)
    
    band['MNDWI'] = (band['green'] - band['swir16']) / (band['green'] + band['swir16'] + eps)
    
    band['NDVI'] =(band['nir']-band['red'])/(band['nir']+band['red'])
    
    evi = 2.5 * (band['nir'] - band['red']) / (band['nir'] + 6*band['red'] - 7.5*band['blue'] + 1)
    band['EVI'] = evi
     
    savi = 1.5 * (band['nir'] - band['red']) / (band['nir'] + band['red'] + 0.5)
    band['SAVI'] = savi
    
    nmdi = (band['nir'] - band['swir16']) / (band['nir'] + band['swir22'] - band['swir16'] + eps)
    band['NMDI'] = nmdi

    fai = band['nir'] - (band['red'] + (band['swir16'] - band['red']) * (band['nir'] - band['red']) / (band['swir16'] - band['red'] + 1e-6))
    band['FAI'] = fai

    turbidity = band['green'] / (band['swir16']+eps)
    band['turbidity'] = turbidity

    ndwi = (band['green'] - band['nir'])/(band['green'] + band['nir'])
    band['NDWI'] = ndwi         

    red_green = band['red'] / (band['green']+eps)
    band['red_green'] = red_green                

    swir_nir = band['swir16'] / (band['nir']+eps)
    band['swir_nir'] = swir_nir
                    
    swir2_nir = band['swir22'] / (band['nir']+eps)
    band['swir2_nir'] = swir2_nir

    band['NDTI'] = (band['red'] - band['green']) / (band['red'] + band['green'] + eps)

    band['BSI'] = ((band['swir16'] + band['red']) - 
               (band['nir'] + band['blue'])) / (
               (band['swir16'] + band['red']) + 
               (band['nir'] + band['blue']) + eps)

    band['AWEI'] = 4*(band['green'] - band['swir16']) - \
               (0.25*band['nir'] + 2.75*band['swir22'])

    band['SI'] = band['swir16'] / (band['green'] + eps)

    
    return band

In [ ]:
landsat_train_fin = create_indices(landsat_train_features)

In [ ]:
display(landsat_train_features.head())
display(landsat_train_fin.head())

In [ ]:
landsat_train_fin = pd.concat(
    [Water_Quality_df.reset_index(drop=True),
     landsat_train_fin.reset_index(drop=True)],
    axis=1
)


In [ ]:
display(landsat_train_fin.head())

In [ ]:
landsat_train_fin.to_csv("/tmp/landsat_features_training_v2.csv",index = False)

In [ ]:
session.sql(f"""
    PUT file://landsat_extraction_training.csv
    snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

In [ ]:
session.sql(f"""
    PUT file:///tmp/landsat_features_training_v2.csv
    snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved! Refresh the browser to see the files in the sidebar")


### Extracting features for the validation dataset

In [11]:
Validation_df=pd.read_csv('submission_template.csv')
display(Validation_df.head())

In [ ]:

val_features_path = "landsat_extraction_val.csv"
batch_size = 50

if os.path.exists(val_features_path):
    existing = pd.read_csv(val_features_path)
    processed_ids = set(existing.index)  # o mejor usar un ID único si lo tienes
else:
    existing = pd.DataFrame()
    processed_ids = set()

results = []

for start in range(0, len(Validation_df), batch_size):
    end = start + batch_size
    batch = Validation_df.iloc[start:end]

    print(f"🚀 Processing rows {start} to {end}")

    for idx, row in batch.iterrows():
        result = safe_compute(row)
        if result is not None:
            pd.DataFrame([result]).to_csv(
                val_features_path,
                mode='a',
                header=not os.path.exists(val_features_path),
                index=False
            )

    print("✅ Batch saved")

In [ ]:
# Extract band values from Landsat for submission dataset
val_features_path = "landsat_features_validation_v2.csv"

print("🚀 Running Landsat feature extraction for validation data...")
landsat_val_features = Validation_df.progress_apply(compute_Landsat_values, axis=1)
landsat_val_features.to_csv(val_features_path, index=False)

In [ ]:
session.sql(f"""
    PUT file://landsat_extraction_val.csv
    snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

In [ ]:
landsat_val_features = pd.read_csv('landsat_extraction_val.csv')
landsat_val_fin = create_indices(landsat_val_features)

In [ ]:
landsat_val_fin = pd.concat(
    [Validation_df.reset_index(drop=True),
     landsat_val_fin.reset_index(drop=True)],
    axis=1
)

In [ ]:
landsat_val_fin.to_csv("/tmp/landsat_features_validation_v2.csv",index = False)

In [ ]:
session.sql(f"""
    PUT file:///tmp/landsat_features_validation_v2.csv
    snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved! Refresh the browser to see the files in the sidebar")
